# Deepfake Audio Detection — end-to-end notebook

**MARS Open Projects 2026 · Problem Statement 2.** Classify speech as **Genuine (Human)** or **Deepfake (AI-Generated)**.

This notebook drives the same library code used in `src/deepfake_audio/` so the notebook, the CLI scripts and the Streamlit app all share one implementation.

**Pipeline:** load audio → trim & fix length → log-mel features (+SpecAugment) → CNN/LCNN → train with EER validation → evaluate (accuracy, EER, F1, confusion matrix) → single-file inference.

**Verification targets:** accuracy ≥ 80%, EER ≤ 12%, F1 ≥ 80%, per-class accuracy ≥ 75%.


## 0. Setup


In [ ]:
# Run once. On Colab/Kaggle keep the CPU/GPU torch wheel appropriate to the runtime.
# !pip install -r requirements.txt && pip install -e .


In [ ]:
import sys; sys.path.insert(0, 'src')
from deepfake_audio.config import load_config
from deepfake_audio.utils.seed import set_seed
cfg = load_config('config/config.yaml')
set_seed(cfg.seed)
print('device:', cfg.device, '| model:', cfg.model.name, '| features:', cfg.features.type)


## 1. Data

Download the Fake-or-Real dataset (see `data/README.md`) and point `data.root` in `config/config.yaml` at the `for-norm` folder. The loader expects `real/` and `fake/` sub-folders under each split.


In [ ]:
from deepfake_audio.data.dataset import list_files
train_files = list_files(cfg.data.root, cfg.data.train_split, cfg.data.real_dirname, cfg.data.fake_dirname)
print('training files found:', len(train_files))
print('example:', train_files[0] if train_files else 'NONE — check data/README.md')


## 2. Feature extraction

Each utterance becomes a per-utterance-normalised log-mel spectrogram of shape `[n_mels, n_frames]`. Visualise one genuine and one deepfake clip.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from deepfake_audio.utils.audio import load_audio, trim_silence, crop_or_pad
from deepfake_audio.features.extract import waveform_to_feature

def show(path, title):
    y = load_audio(path, cfg.audio.sample_rate)
    y = crop_or_pad(trim_silence(y, cfg.audio.top_db), int(cfg.audio.duration_sec*cfg.audio.sample_rate))
    feat = waveform_to_feature(y, cfg)
    plt.figure(figsize=(6,3)); plt.imshow(feat, origin='lower', aspect='auto')
    plt.title(title); plt.xlabel('frames'); plt.ylabel('mel'); plt.colorbar(); plt.show()

# genuine = label 1, deepfake = label 0
g = [p for p,l in train_files if l==1][:1]
d = [p for p,l in train_files if l==0][:1]
if g: show(g[0], 'Genuine (human)')
if d: show(d[0], 'Deepfake (AI-generated)')


## 3. Train

`train()` builds the dataloaders and model, trains with Adam + cosine LR, validates the **EER** every epoch, calibrates the decision threshold at the EER operating point, and checkpoints the best model to `models/best_model.pt`.


In [ ]:
from deepfake_audio.training import train
summary = train('config/config.yaml')
print('best val EER: %.2f%%' % (summary['best_eer']*100))
print('checkpoint:', summary['checkpoint'])


## 4. Evaluate on the test split

Writes `reports/metrics.md` and `reports/figures/confusion_matrix.png`, and checks every value against the competition thresholds.


In [ ]:
from deepfake_audio.training import evaluate
metrics = evaluate('models/best_model.pt', split=cfg.data.test_split)
metrics


## 5. Inference on a new file

The same path the Streamlit app uses. Returns the label and a confidence score.


In [ ]:
from deepfake_audio.inference import load_model, predict_file
model, cfg2, thr = load_model('models/best_model.pt')
# result = predict_file('path/to/your_sample.wav', model, cfg2, thr)
# print(result)


## 6. Web app & deliverables

- Launch the hosted demo locally: `streamlit run app/streamlit_app.py`
- Trained model: `models/best_model.pt`
- Test new audio from the CLI: `python scripts/predict.py --audio sample.wav`
- Performance report: `reports/metrics.md` · confusion matrix: `reports/figures/`
- Architecture & sequence diagrams: `docs/`
